# UN member-state Wikipedia network

Builds a directed, unweighted network where each node is a UN member state
(Wikipedia's `Category:Member_states_of_the_United_Nations`) and an edge
A &rarr; B exists when the running text of A's article links to B's article.

Same shape and spirit as the course's Marvel character network
(`week1_edges.tsv` / `week1_nodes.tsv`), but built here from scratch against
the live English Wikipedia rather than loaded from a provided snapshot --
every step below is deliberate and explained inline, because the choices
made along the way (what counts as a "real" link, which of two overlapping
articles represents one country, etc.) are exactly the kind of thing this
exercise is testing.

Outputs (regenerated by Restart Kernel &amp; Run All):
- `data/un_nodes.tsv`
- `data/un_edges.tsv`
- `data/raw_wikitext/<title>.txt` (cached wikitext, so reruns are fast)


In [1]:
import os
import re
import time
from collections import Counter
from datetime import date

import requests
import pandas as pd
import networkx as nx

# --- Configuration -----------------------------------------------------
# A descriptive, non-personal User-Agent, per Wikimedia's API etiquette
# (https://meta.wikimedia.org/wiki/User-Agent_policy). This notebook may end
# up in a public repo, so it names the project rather than a person.
USER_AGENT = "UNNetworkCrawler/1.0 (student project, see repo README)"
API_URL = "https://en.wikipedia.org/w/api.php"

# Be a polite API citizen: pause after every *actual* network call (not
# after cache hits), and retry transient failures with backoff instead of
# giving up on the first hiccup.
REQUEST_DELAY_SECONDS = 0.4
MAX_RETRIES = 6

CATEGORY = "Category:Member_states_of_the_United_Nations"

DATA_DIR = "data"
RAW_WIKITEXT_DIR = os.path.join(DATA_DIR, "raw_wikitext")
NODES_PATH = os.path.join(DATA_DIR, "un_nodes.tsv")
EDGES_PATH = os.path.join(DATA_DIR, "un_edges.tsv")

os.makedirs(RAW_WIKITEXT_DIR, exist_ok=True)

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})

# A 429 means the *server* is asking us to slow down -- a couple of seconds
# of local backoff isn't enough to recover from that, so a 429 gets a much
# longer, growing wait (honoring Retry-After when the server sends one),
# and it also raises the steady-state delay used by every call after it for
# the rest of the run, since one 429 usually means more are coming if we
# keep going at the same rate.
_current_delay = REQUEST_DELAY_SECONDS


def api_get(params, max_retries=MAX_RETRIES):
    """GET against the MediaWiki API with retries, backoff, and a polite delay.

    Returns the parsed JSON body, or None (after printing why) if every
    retry failed -- callers must check for None so failures are visible
    instead of disappearing silently.
    """
    global _current_delay
    params = dict(params, format="json")
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = SESSION.get(API_URL, params=params, timeout=30)
            if resp.status_code == 429:
                retry_after = resp.headers.get("Retry-After")
                wait = float(retry_after) if retry_after else max(10.0, _current_delay * 8)
                _current_delay = min(_current_delay * 1.5, 3.0)
                print(f"  [429 rate limited, attempt {attempt}/{max_retries}] "
                      f"sleeping {wait:.1f}s (steady-state delay now {_current_delay:.2f}s)")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            time.sleep(_current_delay)
            return resp.json()
        except requests.RequestException as exc:
            last_error = exc
            wait = max(5.0, _current_delay) * (2 ** (attempt - 1))
            print(f"  [retry {attempt}/{max_retries}] {exc} -- backing off {wait:.1f}s")
            time.sleep(wait)
    print(f"FAILED after {max_retries} attempts: params={params!r} error={last_error}")
    return None


def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def safe_filename(title):
    """Canonical titles double as cache filenames; a few characters are
    unsafe across filesystems even though they're legal in a wiki title."""
    return re.sub(r'[\/\\:*?"<>|]', "_", title) + ".txt"

## Step 1 -- the node set: querying the UN member-states category

We don't type the list of 193 countries from memory -- Wikipedia's own
category membership is the ground truth, and it can (and does) contain a
couple of stray non-country pages that need a human's judgment, not a
guess. `cmtype=page` already excludes subcategories, but it does **not**
exclude ordinary articles that happen to be miscategorized (overview pages,
duplicate articles about the same country, etc.), so we fetch, print
everything, and only then decide what -- if anything -- doesn't belong.


In [2]:
def fetch_category_members(category, limit=500):
    members = []
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": category,
        "cmlimit": limit,
        "cmtype": "page",
    }
    while True:
        data = api_get(params)
        if data is None:
            raise RuntimeError(f"Could not fetch category members for {category!r}")
        members.extend(m["title"] for m in data["query"]["categorymembers"])
        if "continue" in data:
            params.update(data["continue"])
        else:
            break
    return members


raw_category_members = sorted(fetch_category_members(CATEGORY))
print(f"Fetched {len(raw_category_members)} pages from {CATEGORY}")
print(f"(193 expected -- {'MATCH' if len(raw_category_members) == 193 else 'MISMATCH, see below'})")
for title in raw_category_members:
    print(" ", title)

Fetched 195 pages from Category:Member_states_of_the_United_Nations
(193 expected -- MISMATCH, see below)
  Afghanistan
  Albania
  Algeria
  Andorra
  Angola
  Antigua and Barbuda
  Argentina
  Armenia
  Australia
  Austria
  Azerbaijan
  Bahrain
  Bangladesh
  Barbados
  Belarus
  Belgium
  Belize
  Benin
  Bhutan
  Bolivia
  Bosnia and Herzegovina
  Botswana
  Brazil
  Brunei
  Bulgaria
  Burkina Faso
  Burundi
  Cambodia
  Cameroon
  Canada
  Cape Verde
  Central African Republic
  Chad
  Chile
  China
  Colombia
  Comoros
  Costa Rica
  Croatia
  Cuba
  Cyprus
  Czech Republic
  Democratic Republic of the Congo
  Denmark
  Djibouti
  Dominica
  Dominican Republic
  Ecuador
  Egypt
  El Salvador
  Equatorial Guinea
  Eritrea
  Estonia
  Eswatini
  Ethiopia
  Federated States of Micronesia
  Fiji
  Finland
  France
  Gabon
  Georgia (country)
  Germany
  Ghana
  Greece
  Grenada
  Guatemala
  Guinea
  Guinea-Bissau
  Guyana
  Haiti
  Honduras
  Hungary
  Iceland
  India
  Indonesia


### Discrepancy found

The live query above currently returns more than 193 pages. Printing the
full list surfaces the stray entries rather than guessing at them:

- **"Member states of the United Nations"** -- the category's own
  overview/list article (the same class of stray the assignment warns
  about, e.g. "List of members of the United Nations"). It's categorized
  alongside the countries it lists, but it isn't one.
- **"Kingdom of the Netherlands"** -- a second, separate article (Wikidata
  `Q29999`, *not* a redirect) about the same UN seat that "Netherlands"
  (`Q55`) also covers. The Kingdom technically holds the UN seat, but every
  other country in this category is titled with its short common name
  (France, not "French Republic"; Kiribati, not "Republic of Kiribati"), so
  keeping *both* Netherlands articles would double-count one seat as two
  nodes. We keep **"Netherlands"**, for consistency with every other entry,
  and drop the long-form duplicate.

The second one is a genuine judgment call, not a mechanical fix -- it's
flagged again in the closing summary for a human to double-check.

Both exclusions are hardcoded below with their reasoning, deliberately
*not* inferred algorithmically. If a future rerun of this notebook turns up
a different count, the assertion right after fails loudly instead of
silently shipping a wrong node set.


In [3]:
# Hardcoded, reasoned exclusions -- see markdown above. Anything else that
# shows up here in a future run is a NEW discrepancy needing the same
# manual scrutiny, not an automatic pass-through.
EXCLUDE_TITLES = {
    "Member states of the United Nations",  # the category's own overview article
    "Kingdom of the Netherlands",           # duplicate of "Netherlands" (Q29999 vs Q55)
}

node_titles = sorted(set(raw_category_members) - EXCLUDE_TITLES)

if len(node_titles) != 193:
    print("Node count did not resolve to 193 after known exclusions -- STOPPING for review.")
    print(f"Got {len(node_titles)}. Full list:")
    for t in node_titles:
        print(" ", t)
    raise AssertionError(
        f"Expected 193 UN member states after exclusions, got {len(node_titles)}. "
        "Inspect the list above and update EXCLUDE_TITLES with reasoning, not a guess."
    )

print(f"Node set resolved to {len(node_titles)} titles after excluding {EXCLUDE_TITLES}.")

Node set resolved to 193 titles after excluding {'Member states of the United Nations', 'Kingdom of the Netherlands'}.


## Step 2 -- canonicalize titles via redirects

Category membership gives us titles *as originally linked into the
category*, which isn't always an article's current canonical title (old
aliases can end up redirected elsewhere over time). We resolve every title
through MediaWiki's own normalization/redirect chain once here, and reuse
the exact same mechanism again in Step 4 for link targets extracted from
article text -- a link target can itself be written as an old alias
(`[[Burma]]` &rarr; `Myanmar`), not just the titles we started from.


In [4]:
def resolve_titles(titles, batch_size=50):
    """Resolve titles through MediaWiki normalization + redirects.

    Returns {original_title: canonical_title}. Titles the API reports as
    missing are printed (not silently dropped) so they can be investigated.
    """
    resolved = {t: t for t in titles}
    for batch in chunked(sorted(set(titles)), batch_size):
        data = api_get({
            "action": "query",
            "titles": "|".join(batch),
            "redirects": 1,
        })
        if data is None:
            print(f"FAILED to resolve redirects for batch: {batch}")
            continue
        q = data["query"]
        normalized = {n["from"]: n["to"] for n in q.get("normalized", [])}
        redirected = {r["from"]: r["to"] for r in q.get("redirects", [])}
        for t in batch:
            cur = normalized.get(t, t)
            cur = redirected.get(cur, cur)
            resolved[t] = cur
        for p in q.get("pages", {}).values():
            if "missing" in p:
                print(f"  MISSING PAGE: {p.get('title')}")
    return resolved


title_to_canonical = resolve_titles(node_titles)
canonical_nodes = sorted(set(title_to_canonical.values()))

changed = {k: v for k, v in title_to_canonical.items() if k != v}
if changed:
    print(f"{len(changed)} title(s) redirected to a different canonical form:")
    for k, v in sorted(changed.items()):
        print(f"  {k!r} -> {v!r}")

if len(canonical_nodes) != len(node_titles):
    print("Redirect resolution collapsed two distinct category entries onto the same "
          "canonical article -- STOPPING for review.")
    counts = Counter(title_to_canonical.values())
    print("Canonical titles hit more than once:", {k: v for k, v in counts.items() if v > 1})
    raise AssertionError("Node count changed after redirect resolution -- inspect above.")

print(f"{len(canonical_nodes)} canonical node titles.")

NODE_SET = set(canonical_nodes)  # membership test used throughout for edges

193 canonical node titles.


## Step 3 -- fetch raw wikitext (not the rendered link index)

This is the most important methodological choice in the notebook, worth
stating explicitly: **we deliberately fetch
`action=query&prop=revisions&rvprop=content` (raw wikitext) instead of
`prop=links` or the rendered HTML.**

The rendered link index -- and the rendered HTML -- includes links injected
by shared navigation templates: the "Member states of the United Nations"
navbox appears at the bottom of essentially every country's article and
would silently wire every country to every other country, and "Foreign
relations of X" sidebars add more templated cross-links on top of that.
None of that reflects an editor choosing, in *that specific article's own
prose*, to mention another country -- counting it would make the network
artificially near-complete (close to a clique), and its degree distribution
and community structure would say more about which templates exist than
about how countries actually reference each other.

Raw wikitext only contains the markup the article's own editors typed.
Regex-extracting `[[...]]` links from *that* gives edges that mean what we
want: "this article's own text links to that one." Fetches are cached to
disk per title, so re-running the notebook while debugging doesn't
re-fetch 193 articles or hammer the API.


In [5]:
def fetch_wikitext(title):
    """Return raw wikitext for `title`, using an on-disk cache.

    A cache hit skips the network entirely, so repeated runs while
    debugging don't re-fetch 193 articles every time.
    """
    cache_path = os.path.join(RAW_WIKITEXT_DIR, safe_filename(title))
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            return f.read()

    data = api_get({
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "titles": title,
    })
    if data is None:
        return None
    page = next(iter(data["query"]["pages"].values()))
    if "missing" in page:
        print(f"FAILED (missing page): {title!r}")
        return None
    try:
        wikitext = page["revisions"][0]["slots"]["main"]["*"]
    except (KeyError, IndexError):
        print(f"FAILED (no revision content): {title!r}")
        return None

    with open(cache_path, "w", encoding="utf-8") as f:
        f.write(wikitext)
    return wikitext


wikitext_by_title = {}
failed_fetches = []
for i, title in enumerate(canonical_nodes, 1):
    wt = fetch_wikitext(title)
    if wt is None:
        failed_fetches.append(title)
    else:
        wikitext_by_title[title] = wt
    if i % 25 == 0 or i == len(canonical_nodes):
        print(f"  fetched {i}/{len(canonical_nodes)}")

print(f"\n{len(wikitext_by_title)} articles fetched, {len(failed_fetches)} failed.")
if failed_fetches:
    print("FAILED titles (need manual attention):", failed_fetches)

  fetched 25/193
  fetched 50/193
  fetched 75/193
  fetched 100/193
  fetched 125/193
  fetched 150/193
  fetched 175/193
  fetched 193/193

193 articles fetched, 0 failed.


## Step 4 -- extract links from wikitext

Rules, applied in order:

1. Regex out every `[[target]]` / `[[target|display]]` span; keep only
   `target`.
2. Drop a trailing `#Section` from the target (same-page anchors aren't
   cross-article links; a bare `[[#Section]]` becomes an empty target and
   is dropped).
3. Drop non-article namespaces (`File:`, `Category:`, `Template:`,
   `Help:`, ...) and interwiki-language prefixes (`de:`, `fr:`, ...) --
   these aren't links to other articles in this network.
4. Resolve the surviving target against a redirect map, because a link
   target may be written as an alias of a node (`[[Burma]]` &rarr;
   `Myanmar`) rather than its canonical form. Built the efficient way: a
   handful of `prop=redirects` calls asking, for each of the 193 nodes,
   "what pages redirect to *this* one" -- rather than one call per
   distinct extracted target asking "what does *this* resolve to". A
   country article can easily contain a couple hundred links, so across
   193 articles the set of distinct targets runs into the thousands;
   resolving each of those individually (as Step 2 does for the 193 node
   titles themselves) would mean hundreds of batched API calls just to
   learn that almost all of them aren't countries. Asking the question in
   reverse needs only ~4 calls (193 nodes / 50 per batch) and gives the
   exact same membership answer, since we only ever care whether a target
   is a known alias of one of the 193 nodes -- never what a non-country
   target actually resolves to.
5. Keep the edge only if the resolved target is one of the 193 canonical
   node titles (implicit in step 4: a target not found in the alias map
   isn't a country page at all). Links to cities, people, organizations,
   wars, etc. are dropped -- they aren't nodes in this network.
6. Drop self-loops (an article linking to itself via an alias).
7. De-duplicate repeated links between the same ordered pair to a single
   edge -- this is an unweighted network.

One limitation of the reverse-lookup approach: it won't follow a *double*
redirect (alias X &rarr; alias Y &rarr; node Z) if X doesn't redirect
straight to Z. Wikipedia's own maintenance bots actively hunt down and fix
double redirects because they break normal reader navigation too, so
they're rare enough in practice not to be worth the extra API cost of
chasing them here.


In [6]:
WIKILINK_RE = re.compile(r'\[\[([^\[\]\n]+)\]\]')

# Non-article namespaces: links into these aren't links to other country
# articles, regardless of what they point at.
NON_ARTICLE_NAMESPACES = {
    "file", "image", "category", "template", "help", "portal", "wikipedia", "wp",
    "user", "user talk", "talk", "special", "mediawiki", "module", "draft",
    "timedtext", "book", "education program", "gadget", "gadget definition",
    "category talk", "template talk", "module talk", "file talk", "portal talk",
    "draft talk", "timedtext talk", "wikipedia talk",
}

# Common interwiki-language / sister-project prefixes. Not an exhaustive
# ISO-639 list -- country articles essentially never hand-write interwiki
# links inline any more (Wikidata sitelinks handle that job now), so this
# is a defensive net for a rare leftover, not a load-bearing filter.
INTERWIKI_PREFIXES = {
    "en", "de", "fr", "es", "it", "pt", "nl", "ru", "zh", "ja", "ko", "ar", "he",
    "hi", "bn", "id", "tr", "pl", "sv", "no", "da", "fi", "el", "cs", "sk", "hu",
    "ro", "bg", "uk", "vi", "th", "fa", "ur", "sw", "simple", "commons", "meta",
    "species", "wikidata", "wiktionary", "wikibooks", "wikinews", "wikiquote",
    "wikisource", "wikiversity", "wikivoyage", "incubator", "mw",
}


def normalize_target(raw_target):
    """MediaWiki-normalize a raw link target for matching against titles."""
    t = raw_target.strip().replace("_", " ")
    t = t.split("|", 1)[0].strip()      # piped link: keep only the target
    t = t.split("#", 1)[0].strip()      # drop a trailing #Section anchor
    if t.startswith(":"):
        t = t[1:].strip()               # "[[:Category:X]]" links TO the page
    if not t:
        return None
    prefix = t.split(":", 1)[0].strip().lower() if ":" in t else None
    if prefix and (prefix in NON_ARTICLE_NAMESPACES or prefix in INTERWIKI_PREFIXES):
        return None
    return t[0].upper() + t[1:]  # enwiki capitalizes a title's first letter


def extract_link_targets(wikitext):
    targets = []
    for raw in WIKILINK_RE.findall(wikitext):
        norm = normalize_target(raw)
        if norm:
            targets.append(norm)
    return targets


links_by_title = {title: extract_link_targets(wt) for title, wt in wikitext_by_title.items()}

all_raw_targets = sorted({t for targets in links_by_title.values() for t in targets})
print(f"{len(all_raw_targets)} distinct link targets across {len(links_by_title)} articles "
      "(before redirect resolution / node-set filtering) -- almost all of these will turn "
      "out not to be countries at all, which is exactly why we resolve in reverse below.")


def fetch_redirects_into(titles, batch_size=50, rdlimit=500):
    """For each of `titles`, fetch every page that redirects TO it.

    Returns {alias_title: canonical_title}, covering every known alias of
    every title in `titles` plus each title mapped to itself. See the
    markdown above for why this reverse direction is used instead of
    resolving each distinct extracted target individually.
    """
    alias_to_canonical = {t: t for t in titles}
    for batch in chunked(sorted(set(titles)), batch_size):
        data = api_get({
            "action": "query",
            "titles": "|".join(batch),
            "prop": "redirects",
            "rdlimit": rdlimit,
        })
        if data is None:
            print(f"FAILED to fetch redirects-into for batch: {batch}")
            continue
        for page in data["query"]["pages"].values():
            canonical = page.get("title")
            if canonical is None:
                continue
            for r in page.get("redirects", []):
                alias_to_canonical[r["title"]] = canonical
    return alias_to_canonical


ALIAS_TO_NODE = fetch_redirects_into(canonical_nodes)
print(f"{len(ALIAS_TO_NODE)} titles (193 canonical nodes + every known alias of each) "
      "resolve to a UN member state.")

edges = set()
for source, targets in links_by_title.items():
    for raw_target in targets:
        target = ALIAS_TO_NODE.get(raw_target)
        if target is None:
            continue  # not a known alias of any of the 193 nodes -- not an edge here
        if target == source:
            continue  # drop self-loops (e.g. an alias pointing back at itself)
        edges.add((source, target))

edges = sorted(edges)
print(f"{len(edges)} directed edges after namespace/redirect filtering, "
      "node-set restriction, self-loop removal, and de-duplication.")

84701 distinct link targets across 193 articles (before redirect resolution / node-set filtering) -- almost all of these will turn out not to be countries at all, which is exactly why we resolve in reverse below.


2193 titles (193 canonical nodes + every known alias of each) resolve to a UN member state.
2193 directed edges after namespace/redirect filtering, node-set restriction, self-loop removal, and de-duplication.


### A deliberate inclusion worth flagging: infobox `neighbours` fields

Some country infoboxes carry a field like
`| neighbours = [[Germany]], [[Belgium]], ...` written directly into that
specific article's own wikitext -- it is **not** injected by a shared
template the way the UN navbox is. Under the rule from Step 3 ("count what
the article's own editors wrote"), these links count as genuine edges: the
infobox is part of the article's own content, on the same footing as any
other inline link, even though it renders as a structured field rather
than a sentence.

This is a choice, not an accident, in the same spirit as the navbox
exclusion above -- reasonable people could draw the infobox/navbox line
differently (e.g. treat *all* infobox fields as "structured metadata" and
exclude them too). We include them because they're per-article content,
not per-category boilerplate. Flagging it here so it's an informed choice
rather than a silent one.


## Step 6 -- node metadata

For each of the 193 canonical titles we fetch a one-line description
(`prop=extracts&exintro=1&exsentences=1&explaintext=1` -- `exsentences=1`
trims the intro extract to its first sentence, matching the style of
`week1_nodes.tsv`'s descriptions) and a Wikidata id (`prop=pageprops`,
reading `wikibase_item`). We don't restrict `ppprop`, so the same call also
lets us check for a `disambiguation` flag as a sanity check on the node
set. Batched at 50 titles per call rather than one request per country.

One MediaWiki quirk worth naming: `prop=extracts` silently caps itself at
20 results per request (`limits.extracts == 20` in the response) no matter
how many titles are asked for or what `exlimit` is set to -- it does *not*
error on the other 30, it just omits their `extract` field and returns a
`continue` token. `prop=pageprops` has no such cap, so a naive read would
get Wikidata ids for all 50 titles but descriptions for only the first 20
and never notice. We follow `continue` until it's exhausted so every title
in the batch actually gets its description.


In [7]:
def fetch_metadata(titles, batch_size=50):
    """Returns {title: {'description', 'wikidata_id', 'disambiguation'}}.

    See the markdown above: prop=extracts caps at 20 results per request
    regardless of batch size, signaling the rest via a `continue` token
    rather than an error -- so each batch of up to 50 titles may take a
    couple of follow-up requests to fully drain.
    """
    meta = {}
    for batch in chunked(sorted(titles), batch_size):
        params = {
            "action": "query",
            "prop": "extracts|pageprops",
            "exintro": 1,
            "exsentences": 1,
            "explaintext": 1,
            "titles": "|".join(batch),
        }
        while True:
            data = api_get(params)
            if data is None:
                print(f"FAILED metadata batch: {batch}")
                break
            for page in data["query"]["pages"].values():
                title = page.get("title")
                if "missing" in page:
                    print(f"  MISSING PAGE for metadata: {title!r}")
                    continue
                entry = meta.setdefault(
                    title, {"description": "", "wikidata_id": "", "disambiguation": False}
                )
                props = page.get("pageprops", {})
                description = page.get("extract", "").strip().replace("\n", " ")
                if description:
                    entry["description"] = description
                if props.get("wikibase_item"):
                    entry["wikidata_id"] = props["wikibase_item"]
                if "disambiguation" in props:
                    entry["disambiguation"] = True
            cont = data.get("continue")
            if not cont:
                break
            params = dict(params, **cont)
    return meta


metadata_by_title = fetch_metadata(canonical_nodes)

missing_wikidata = [t for t in canonical_nodes if not metadata_by_title.get(t, {}).get("wikidata_id")]
missing_description = [t for t in canonical_nodes if not metadata_by_title.get(t, {}).get("description")]
disambiguation_hits = [t for t in canonical_nodes if metadata_by_title.get(t, {}).get("disambiguation")]

if missing_wikidata:
    print("No Wikidata id found for:", missing_wikidata)
if missing_description:
    print("No description extract found for:", missing_description)
if disambiguation_hits:
    print("Resolved to what MediaWiki flags as a DISAMBIGUATION page -- needs a human look:",
          disambiguation_hits)

  [429 rate limited, attempt 1/6] sleeping 35.0s (steady-state delay now 0.60s)


  [429 rate limited, attempt 1/6] sleeping 50.0s (steady-state delay now 0.90s)


## Step 7 -- build the graph

All 193 nodes are added to the `DiGraph` explicitly *before* any edges, so
a country with zero in/out links (an isolate) is preserved rather than
silently vanishing because it never appears in an edge tuple -- the same
gotcha week 1's Marvel network calls out with its 17 edge-less characters.


In [8]:
G = nx.DiGraph()
G.add_nodes_from(canonical_nodes)
G.add_edges_from(edges)

isolates = sorted(nx.isolates(G))

components = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
giant = components[0] if components else set()
outside_giant = sorted(set(canonical_nodes) - giant)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Isolates ({len(isolates)}): {isolates}")
print(f"Weakly connected components: {len(components)} (giant component has {len(giant)} nodes)")
print(f"Countries outside the giant weakly connected component ({len(outside_giant)}): {outside_giant}")

Nodes: 193
Edges: 2193
Isolates (0): []
Weakly connected components: 1 (giant component has 193 nodes)
Countries outside the giant weakly connected component (0): []


## Step 8 -- write the output files

Same tab-separated, `#`-comment-headered shape as `week1_nodes.tsv` /
`week1_edges.tsv`, so both datasets load with the same pandas incantation.


In [9]:
SNAPSHOT_DATE = date.today().isoformat()

nodes_rows = []
for title in canonical_nodes:
    meta = metadata_by_title.get(title, {})
    nodes_rows.append({
        "node_id": title.replace(" ", "_"),
        "name": title,
        "wikidata_id": meta.get("wikidata_id", ""),
        "url": "https://en.wikipedia.org/wiki/" + title.replace(" ", "_"),
        "description": meta.get("description", ""),
    })
nodes_df = pd.DataFrame(nodes_rows).sort_values("node_id").reset_index(drop=True)

edges_rows = [
    {"source": s.replace(" ", "_"), "target": t.replace(" ", "_")}
    for s, t in edges
]
edges_df = pd.DataFrame(edges_rows).sort_values(["source", "target"]).reset_index(drop=True)

nodes_header = (
    f"# UN member-state Wikipedia network -- built {SNAPSHOT_DATE}\n"
    f"# All 193 UN member states from Wikipedia's Category:Member_states_of_the_United_Nations\n"
    f"# (redirects resolved; 2 miscategorized non-country pages excluded, see notebook Step 1).\n"
    f"# {len(nodes_df)} nodes. node_id matches un_edges.tsv.\n"
)
edges_header = (
    f"# UN member-state Wikipedia network -- built {SNAPSHOT_DATE}\n"
    f"# Directed, unweighted: edge A -> B when A's article wikitext links to B's,\n"
    f"# counting only inline links from the article's OWN prose/infobox -- links\n"
    f"# injected by shared navigation templates (e.g. the UN navbox) are excluded.\n"
    f"# {len(nodes_df)} nodes, {len(edges_df)} directed edges.\n"
    f"# source\ttarget\n"
)

with open(NODES_PATH, "w", encoding="utf-8") as f:
    f.write(nodes_header)
    nodes_df.to_csv(f, sep="\t", index=False)

with open(EDGES_PATH, "w", encoding="utf-8") as f:
    f.write(edges_header)
    edges_df.to_csv(f, sep="\t", index=False, header=False)

print(f"Wrote {NODES_PATH} ({len(nodes_df)} rows) and {EDGES_PATH} ({len(edges_df)} rows).")

Wrote data/un_nodes.tsv (193 rows) and data/un_edges.tsv (2193 rows).


In [10]:
print("=== Final summary ===")
print(f"Nodes: {G.number_of_nodes()}   Edges: {G.number_of_edges()}")
print(f"Isolates: {len(isolates)} -> {isolates}")
print(f"Outside giant weakly-connected component: {len(outside_giant)} -> {outside_giant}")
print(f"Disambiguation-flagged titles: {disambiguation_hits}")
print(f"Failed wikitext fetches: {failed_fetches}")

=== Final summary ===
Nodes: 193   Edges: 2193
Isolates: 0 -> []
Outside giant weakly-connected component: 0 -> []
Disambiguation-flagged titles: []
Failed wikitext fetches: []


## Summary

Final node/edge counts, isolates, and the giant-component check are all in
the cell output immediately above -- they depend on the live snapshot this
notebook was last run against, so they're read off the output rather than
restated here.

**Data-quality wrinkles worth a second look:**

1. **"Netherlands" vs "Kingdom of the Netherlands"** (Step 1) -- we kept
   the short common name and dropped the long-form duplicate for
   consistency with every other entry, but the UN seat is technically held
   by the Kingdom. Worth confirming this matches how sovereignty should be
   represented for this project.
2. Any title in `disambiguation_hits` above resolved to a page MediaWiki
   itself flags as ambiguous rather than a specific article -- its
   outgoing edges and description would be meaningless and should be
   re-pointed by hand.
3. Any title in `failed_fetches` above didn't get wikitext at all
   (network/API failure) and is missing its outgoing edges entirely --
   rerunning the notebook only re-fetches the failed ones (the cache
   short-circuits the rest), or it can be investigated manually.
4. Isolates, if any, are most likely genuinely insular countries whose
   articles happen not to name-check another UN member inline in running
   prose -- e.g. small Pacific/Caribbean island states whose "foreign
   relations" content leans on organizations ("the United Nations", "the
   Commonwealth") rather than naming individual countries by name. Worth
   eyeballing the actual isolate list above rather than assuming that's
   the whole story.
